# Employment status rolling average: compare (ticket 2260)

Loads the results written by `experiment.ipynb` and applies the win rule against the current setup. Read only.

**Win rule** (test A, and the variant must not lose to the current setup on test B):
- at least `MAE_WIN_PP` lower mean absolute error and no jumpier than current, or
- at least `MAE_SMOOTHER_WIN_PP` lower mean absolute error and jumpiness at least `SMOOTHER_BY` lower.

Every row must have a prediction (coverage 1.0). Jumpiness is workplace level; the aggregate level (service by region by job role) is shown beside it.

## 1. Parameters

- `RESULTS_PREFIX` and `RUN_LABEL` as in `experiment.ipynb`.
- The thresholds define the win rule. `SMOOTHER_BY` is the relative drop in jumpiness counted as clearly smoother.
- `LOCAL_ROOT` is for dry runs only.

In [ ]:
RESULTS_PREFIX = "s3://skillsforcare-nonprod/sagemaker-notebooks/empstat-rolling-avg/"
RUN_LABEL = "run1"
CURRENT = "current_6mo"
MAE_WIN_PP = 1.0
MAE_SMOOTHER_WIN_PP = 0.5
SMOOTHER_BY = 0.10
LOCAL_ROOT = None
REPO_DIR = "repo"

## Setup

Reads the repo's `experiments_2260/experiment_utils.py` for the result readers; clone it first if this notebook runs on its own: `git clone --depth 1 --branch 2260-empstat-avg https://github.com/NMDSdevopsServiceAdm/DataEngineering.git repo`.

In [ ]:
import sys
from pathlib import Path

import polars as pl

sys.path.insert(0, str(Path(REPO_DIR).resolve()))

import experiments_2260.experiment_utils as x
from utils.column_names.ind_cqc_pipeline_columns import ModelEvaluationColumns as Eval

LOCAL = LOCAL_ROOT is not None
RESULTS_DIR = (
    str(Path(LOCAL_ROOT) / "results" / RUN_LABEL) if LOCAL else f"{RESULTS_PREFIX.rstrip('/')}/{RUN_LABEL}"
)
MAE = f"pooled_{Eval.mean_absolute_error_pp}"
JUMPINESS = f"jumpiness_workplace_{Eval.mean_period_to_period_change}"
JUMPINESS_AGGREGATE = f"jumpiness_aggregate_{Eval.mean_period_to_period_change}"
COVERAGE = f"coverage_{Eval.coverage}"
pl.Config.set_tbl_cols(40)
pl.Config.set_tbl_rows(100)

## 2. Load results

One row per variant and measure. Test B only has the top variants and the current setup.

In [ ]:
def load_all(prefix: str) -> pl.DataFrame:
    return pl.concat(
        [x.read_result(name, RESULTS_DIR, LOCAL) for name in x.list_results(prefix, RESULTS_DIR, LOCAL)]
    )


def headline(results_df: pl.DataFrame) -> pl.DataFrame:
    return (
        results_df.filter(
            (pl.col(Eval.level).is_in(["pooled", "coverage", "jumpiness_workplace", "jumpiness_aggregate"]))
            & (pl.col(Eval.metric) != Eval.number_of_rows)
        )
        .with_columns((pl.col(Eval.level) + "_" + pl.col(Eval.metric)).alias("measure"))
        .pivot(on="measure", index="variant", values=Eval.value)
    )


test_a_df = load_all("test_a/")
test_b_df = load_all("test_b/")
a_table = headline(test_a_df)
b_table = headline(test_b_df)

## 3. Full table, test A

Every variant: error (pp), Brier, top 1, skill scores, jumpiness (workplace and aggregate) and coverage. Sorted by error.

In [ ]:
a_table.sort(MAE)

## 4. Full table, test B

The same measures on the hidden last 12 months.

In [ ]:
b_table.sort(MAE)

## 5. Win rule

`beats_a`: the test A rule against `CURRENT`. `not_worse_on_b`: test B error no higher than `CURRENT`'s (a variant not run on test B cannot win). `full_coverage`: coverage 1.0. `win` needs all three. Change in error is in pp; negative is better.

In [ ]:
current_a = a_table.filter(pl.col("variant") == CURRENT).row(0, named=True)
current_b = b_table.filter(pl.col("variant") == CURRENT).row(0, named=True)

verdict_df = (
    a_table.join(
        b_table.select("variant", pl.col(MAE).alias("mae_b")), on="variant", how="left"
    )
    .with_columns(
        (pl.col(MAE) - current_a[MAE]).alias("mae_change_pp"),
        (pl.col(JUMPINESS) / current_a[JUMPINESS] - 1).alias("jumpiness_change"),
        (pl.col("mae_b") - current_b[MAE]).alias("mae_b_change_pp"),
    )
    .with_columns(
        (
            ((pl.col("mae_change_pp") <= -MAE_WIN_PP) & (pl.col(JUMPINESS) <= current_a[JUMPINESS]))
            | ((pl.col("mae_change_pp") <= -MAE_SMOOTHER_WIN_PP) & (pl.col("jumpiness_change") <= -SMOOTHER_BY))
        ).alias("beats_a"),
        (pl.col("mae_b_change_pp") <= 0).fill_null(False).alias("not_worse_on_b"),
        (pl.col(COVERAGE) >= 1.0).alias("full_coverage"),
    )
    .with_columns((pl.col("beats_a") & pl.col("not_worse_on_b") & pl.col("full_coverage")).alias("win"))
    .select(
        "variant", "mae_change_pp", "jumpiness_change", "mae_b_change_pp",
        "beats_a", "not_worse_on_b", "full_coverage", "win",
    )
    .sort("mae_change_pp")
)
print(verdict_df)
winners = verdict_df.filter(pl.col("win"))["variant"].to_list()
print("Winners:", winners or "none: no pipeline change")

## 6. Per job role error

Mean absolute error (pp) per job role for the current setup and each variant that passed test A, with the change against current. Negative is better. Use it to spot a variant that helps on average but hurts one role.

In [ ]:
role_df = test_a_df.filter(
    (pl.col(Eval.level) == "job_role") & (pl.col(Eval.metric) == Eval.mean_absolute_error_pp)
).pivot(on=Eval.group, index="variant", values=Eval.value)
shown = [CURRENT] + verdict_df.filter(pl.col("beats_a"))["variant"].to_list()
role_df.filter(pl.col("variant").is_in(shown))

## 7. Per size band error

Mean absolute error (pp) per workplace size band, for every variant. Size is the sum of job role estimates per workplace and date.

In [ ]:
test_a_df.filter(
    (pl.col(Eval.level) == "size_band") & (pl.col(Eval.metric) == Eval.mean_absolute_error_pp)
).pivot(on=Eval.group, index="variant", values=Eval.value).sort("variant")